In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.answer_extractor import AnswerExtractor

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [2]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'separated_targets_20x50x4__26_11/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

In [3]:
mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)
mres_sep = MultiVariantMultiModelResultsAnalyser(p_sep)
mres_code = MultiVariantMultiModelResultsAnalyser(p_code)

Model: 100%|██████████| 20/20 [00:00<00:00, 74.88it/s]
/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/results_analyser/multi_model.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(full_data_dict.values(), keys=full_data_dict.keys(), names=['model', 'old_index'])
Model: 100%|██████████| 20/20 [00:00<00:00, 34.06it/s]
/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/results_analyser/multi_model.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat ope

## Comparison to baseline (GSM8K) questions

In [4]:
drop_standard = mres_standard.get_accuracy_drop_df('main')
drop_standard

accuracy_drop  strict_accuracy_drop
model                     id                                     
google_gemma-2-27b-it     0            0.00                 -0.05
                          1            0.00                  0.00
                          2            0.00                  0.35
                          3            0.00                  0.80
                          4           -0.05                 -0.05
...                                     ...                   ...
mistralai_Mistral-7B-v0_3 45          -0.25                  0.00
                          46          -0.25                  0.00
                          47          -0.05                  0.00
                          48           0.00                  0.00
                          49          -0.05                  0.00

[1000 rows x 2 columns]

In [5]:
drop_code = mres_code.get_accuracy_drop_df('main')

In [6]:
gap_closure_code = drop_standard - drop_code
gap_closure_code.rename(columns={'accuracy_drop': 'gap_closure', 'strict_accuracy_drop': 'strict_gap_closure'}, inplace=True)
gap_closure_code

gap_closure  strict_gap_closure
model                     id                                 
google_gemma-2-27b-it     0          0.00               -0.05
                          1          0.00                0.00
                          2          0.45                0.80
                          3         -0.15                0.65
                          4         -0.15               -0.15
...                                   ...                 ...
mistralai_Mistral-7B-v0_3 45        -0.25                0.00
                          46         0.00                0.00
                          47        -0.35                0.15
                          48         0.00                0.00
                          49         0.00                0.00

[1000 rows x 2 columns]

In [7]:
print(f"Mean Drop (Standard):\n{drop_standard.mean()}")
print(f"\nMean Drop (Code):\n{drop_code.mean()}")
print(f"\nMean Gap Closure:\n{gap_closure_code.mean()}")

Mean Drop (Standard):
accuracy_drop           0.06550
strict_accuracy_drop    0.03245
dtype: float64

Mean Drop (Code):
accuracy_drop           0.02960
strict_accuracy_drop    0.00865
dtype: float64

Mean Gap Closure:
gap_closure           0.0359
strict_gap_closure    0.0238
dtype: float64


In [8]:
from scipy import stats

def report_significance(control_drops, treatment_drops, p_threshold=0.05):
    t_stat, p_val_ttest = stats.ttest_rel(control_drops, treatment_drops, alternative='greater')
    print(f"P-value {p_val_ttest:.5f}")
    print(f"{'STATISTICALLY' if p_val_ttest < p_threshold else 'NOT'} SIGNIFICANT")

print(f"--- Paired T-Test ---")
print("\nLenient accuracy:")
report_significance(drop_standard.accuracy_drop, drop_code.accuracy_drop)

print("\nStrict accuracy:")
report_significance(drop_standard.strict_accuracy_drop, drop_code.strict_accuracy_drop)


--- Paired T-Test ---

Lenient accuracy:
P-value 0.00758
STATISTICALLY SIGNIFICANT

Strict accuracy:
P-value 0.01898
STATISTICALLY SIGNIFICANT
